<a href="https://colab.research.google.com/github/YASHARTH1630/quora-duplicate-question-detection/blob/main/experiment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Dataset

### The dataset consists of pairs of questions along with a binary label indicating whether the two questions are semantically equivalent. Each record contains:
### question1: First question
### question2: Second question
### is_duplicate: Target label (1 = duplicate, 0 = not duplicate)

## The objective is to predict whether two questions express the same intent despite differences in wording.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df=pd.read_csv("train.csv")
df.head(10)

In [ ]:
df.shape

## calculating weight of 0 and 1

In [ ]:
print(df['is_duplicate'].value_counts())
print((df['is_duplicate'].value_counts()/df['is_duplicate'].count())*100)
df['is_duplicate'].value_counts().plot(kind='bar')

In [ ]:

# Repeated questions

qid = pd.Series(df['qid1'].tolist() + df['qid2'].tolist())
print('Number of unique questions',np.unique(qid).shape[0])
x = qid.value_counts()>1
print('Number of questions getting repeated',x[x].shape[0])

In [ ]:

# Repeated questions histogram

plt.hist(qid.value_counts().values,bins=160)
plt.yscale('log')
plt.show()

In [ ]:
df.drop(columns=["id","qid1","qid2"],inplace=True)

In [ ]:
df.drop_duplicates(inplace=True)

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df.dropna(subset=['question1', 'question2'], inplace=True)

In [ ]:
max_len = df['question1'].str.split().str.len().max()
print(max_len)

In [ ]:
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense


# Text Preprocessing

In [ ]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv("train.csv")

df = df[['question1', 'question2', 'is_duplicate']]

df.dropna(inplace=True)

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z0-9 ]', '', text)
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    text = re.sub(r'<.*?>', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['question1'] = df['question1'].apply(clean_text)
df['question2'] = df['question2'].apply(clean_text)

In [ ]:
all_questions = list(df['question1']) + list(df['question2'])

### Tokenizing the word

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()

tokenizer.fit_on_texts(all_questions)

In [ ]:
total_words=len(tokenizer.word_index)+1
total_words

In [ ]:
tokenizer.word_index

### Making the sentence from tokenized words

In [ ]:
q1_seq = tokenizer.texts_to_sequences(df['question1'])

In [ ]:
q2_seq = tokenizer.texts_to_sequences(df['question2'])

### Padding in the sentences to make uniform vector size of sentence

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

max_len = 300

q1_pad = pad_sequences(q1_seq, maxlen=max_len)

q2_pad = pad_sequences(q2_seq, maxlen=max_len)

In [ ]:
q1_pad.shape

In [ ]:
y=df["is_duplicate"] ##Output

## Model Training and Initialisation

### Model Architecture
 #### A Siamese Neural Network was used for duplicate question detection. The architecture processes both questions through the same Embedding and LSTM encoder, ensuring that both inputs are mapped into the same semantic feature space using shared weights. The resulting embeddings are compared using their absolute difference, and the combined features are passed through dense layers to predict whether the questions are duplicates

In [ ]:
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Input,
    Embedding,
    LSTM,
    Dense,
    Dropout,
    Concatenate,
    Lambda
)
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

# INPUT LAYERS
input1 = Input(shape=(max_len,))
input2 = Input(shape=(max_len,))

# EMBEDDING LAYER

embedding = Embedding(
    input_dim=total_words, ##total_words analysised by tokenizer and gave them a number
    output_dim=150
)

x1 = embedding(input1)
x2 = embedding(input2)

# SHARED LSTM

shared_lstm = LSTM(150)

out1 = shared_lstm(x1)
out2 = shared_lstm(x2)

# =========================
# DIFFERENCE LAYER
# =========================
def abs_diff(x):
    return tf.abs(x[0] - x[1])

diff = Lambda(abs_diff)([out1, out2])
def mul_sim(x):
  return tf.multiply(x[0],x[1])
mul  = Lambda(mul_sim)([out1, out2]) ##similarity

merged = Concatenate()([out1, out2, diff,mul])

# DENSE LAYER
dense = Dense(
    128,
    activation='relu'
)(merged)

dense = Dropout(0.3)(dense)
# OUTPUT LAYER
output = Dense(
    1,
    activation='sigmoid'
)(dense)

# FINAL MODEL
model = Model(
    inputs=[input1, input2],
    outputs=output
)

# COMPILE
model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)
# SUMMARY

model.summary()

In [ ]:
## Early Stopping
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=3)

In [ ]:
## Train the model over training data
model.fit(
    [q1_pad, q2_pad],
    y,
    epochs=6,
    batch_size=32,
    callbacks=[early_stopping],
    validation_split=0.2
)

## Validation Accuracy is 85.4%

In [ ]:
## Save the model for prediction
model.save("similiar.h5")

In [ ]:
import pickle

with open("tokenizer1.pkl", "wb") as f:
    pickle.dump(tokenizer, f)